# Lower Volta Final Analysis\nRun Earth Engine exports first, then execute all cells.\n

## 01_install_and_config.py\n

In [ ]:
# 01 — Environment and configuration\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\ntry:\n    from google.colab import drive\n    drive.mount('/content/drive')\n    ROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports')\nexcept Exception:\n    ROOT = Path('data/exports')\n\nOUT = Path('outputs')\nOUT.mkdir(parents=True, exist_ok=True)\n\nEXPECTED = {\n    'study_area_km2': 8938.927,\n    'event_temporary_km2': 5.868,\n    'cumulative_km2': 7.386,\n    'recurrent_km2': 4.487,\n    'sporadic_km2': 2.899,\n    'surface_change_km2': 2032.146,\n    'TP': 278, 'TN': 297, 'FP': 3, 'FN': 22\n}\nprint('Input:', ROOT)\nprint('Output:', OUT)\n

## 02_threshold_benchmark.py\n

In [ ]:
# 02 — Sentinel-1 method benchmark and threshold sensitivity\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nOUT = Path('outputs'); OUT.mkdir(exist_ok=True)\n\nbenchmark = pd.DataFrame({\n    'method':['Otsu VV','Otsu VH','Fixed dual polarisation'],\n    'threshold':['-15.559 dB','-23.572 dB','VV < -17 dB; VH < -23 dB'],\n    'temporary_water_km2':[7.676,7.129,5.868],\n    'diagnostic_f1_pct':[75.7,74.3,70.7]\n})\nbenchmark.to_csv(OUT/'Table3_method_benchmark.csv',index=False)\n\nvv=[-18,-17,-16]; vh=[-24,-23,-22]\nf1=np.array([[63.5,69.4,72.8],[64.1,70.7,74.3],[64.1,71.0,74.9]])\npd.DataFrame(f1,index=vh,columns=vv).to_csv(OUT/'Supplementary_Table_S2_threshold_sensitivity.csv')\n\nfig,ax=plt.subplots(figsize=(6,5))\nbars=ax.bar(benchmark['method'],benchmark['diagnostic_f1_pct'])\nax.set_ylabel('F1-score (%)'); ax.set_ylim(50,100); ax.set_title('(a)',loc='left',fontweight='bold')\nax.tick_params(axis='x',rotation=20); ax.grid(axis='y',alpha=.25)\nfor b,v in zip(bars,benchmark['diagnostic_f1_pct']):\n    ax.text(b.get_x()+b.get_width()/2,v+1,f'{v:.1f}',ha='center',fontweight='bold')\nfig.tight_layout(); fig.savefig(OUT/'Figure5a_method_benchmark.png',dpi=600,bbox_inches='tight'); plt.close(fig)\n\nfig,ax=plt.subplots(figsize=(6,5))\nim=ax.imshow(f1,vmin=60,vmax=80,aspect='auto')\nax.set_xticks(range(3),vv); ax.set_yticks(range(3),vh)\nax.set_xlabel('VV threshold (dB)'); ax.set_ylabel('VH threshold (dB)')\nax.set_title('(b)',loc='left',fontweight='bold')\nfor i in range(3):\n    for j in range(3):\n        ax.text(j,i,f'{f1[i,j]:.1f}',ha='center',va='center',fontweight='bold')\ncb=fig.colorbar(im,ax=ax); cb.set_label('F1-score (%)')\nfig.tight_layout(); fig.savefig(OUT/'Figure5b_threshold_sensitivity.png',dpi=600,bbox_inches='tight'); plt.close(fig)\n\nprint(benchmark)\n

## 03_hydroclimate_stats.py\n

In [ ]:
# 03 — Hydro-climatic correlations\nfrom pathlib import Path\nimport pandas as pd\nfrom scipy.stats import pearsonr, spearmanr\n\nROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')\nOUT=Path('outputs'); OUT.mkdir(exist_ok=True)\np=ROOT/'LV_Hydroclimate_Per_Observation.csv'\nif not p.exists():\n    raise FileNotFoundError(f'Missing {p}; run GEE script 03 first.')\ndf=pd.read_csv(p)\nrows=[]\nfor prefix,label in [('rain_','CHIRPS rainfall'),('runoff_','ERA5-Land runoff')]:\n    for d in [1,3,7,14,30]:\n        c=f'{prefix}{d}d_mm'\n        if c not in df.columns: continue\n        x=df[['temporary_flood_km2',c]].dropna()\n        if len(x)<3: continue\n        pr,pp=pearsonr(x['temporary_flood_km2'],x[c])\n        sr,sp=spearmanr(x['temporary_flood_km2'],x[c])\n        rows.append({'variable':label,'window_days':d,'n':len(x),\n                     'pearson_r':pr,'pearson_p':pp,'spearman_rho':sr,'spearman_p':sp})\ncorr=pd.DataFrame(rows)\ncorr.to_csv(OUT/'Supplementary_Table_S3_hydroclimatic_correlations.csv',index=False)\nprint(corr.to_string(index=False))\n

## 04_agreement_metrics.py\n

In [ ]:
# 04 — Sentinel-1 / Sentinel-2 inter-sensor agreement\nfrom pathlib import Path\nimport pandas as pd\nfrom sklearn.metrics import confusion_matrix\n\nROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')\nOUT=Path('outputs'); OUT.mkdir(exist_ok=True)\np=ROOT/'LV_S1_S2_600_Point_Comparison.csv'\n\nif p.exists():\n    d=pd.read_csv(p)\n    tn,fp,fn,tp=confusion_matrix(d['s2_class'].astype(int),d['s1_class'].astype(int),labels=[0,1]).ravel()\nelse:\n    tp,tn,fp,fn=278,297,3,22\n    print('Point CSV absent: using archived final manuscript confusion counts.')\n\nn=tp+tn+fp+fn\noa=(tp+tn)/n\nprecision=tp/(tp+fp)\nrecall=tp/(tp+fn)\nspecificity=tn/(tn+fp)\nf1=2*precision*recall/(precision+recall)\nomission=fn/(tp+fn)\ncommission=fp/(tp+fp)\nnonf_prod=tn/(tn+fp)\nnonf_user=tn/(tn+fn)\nnonf_omission=fp/(tn+fp)\nnonf_commission=fn/(tn+fn)\npe=((tp+fn)/n)*((tp+fp)/n)+((tn+fp)/n)*((tn+fn)/n)\nkappa=(oa-pe)/(1-pe)\n\nmetrics=pd.DataFrame({\n 'metric':['TP','TN','FP','FN','Total samples','Overall agreement (%)','Flood precision (%)',\n 'Flood recall (%)','Flood specificity (%)','Flood F1-score (%)','Flood omission error (%)',\n 'Flood commission error (%)','Non-flood producer agreement (%)','Non-flood user agreement (%)',\n 'Non-flood omission error (%)','Non-flood commission error (%)','Cohen kappa'],\n 'value':[tp,tn,fp,fn,n,oa*100,precision*100,recall*100,specificity*100,f1*100,\n omission*100,commission*100,nonf_prod*100,nonf_user*100,nonf_omission*100,nonf_commission*100,kappa]\n})\nmetrics.to_csv(OUT/'Table5_inter_sensor_metrics.csv',index=False)\nprint(metrics.to_string(index=False))\n

## 05_tables_figures.py\n

In [ ]:
# 05 — Consolidate tables from Earth Engine exports\nfrom pathlib import Path\nimport pandas as pd\n\nROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')\nOUT=Path('outputs'); OUT.mkdir(exist_ok=True)\n\ndef load(name):\n    p=ROOT/name\n    if not p.exists():\n        print('Missing:',p); return None\n    return pd.read_csv(p)\n\ncore=load('LV_Core_Area_Summary.csv')\nterrain=load('LV_Terrain_Statistics.csv')\nwc=load('LV_WorldCover_Exposure.csv')\nghsl=load('LV_GHSL_Built_Exposure.csv')\nmndwi=load('LV_MNDWI_Sensitivity.csv')\nmonthly=load('LV_CHIRPS_Monthly_2023.csv')\n\nif core is not None: core.to_csv(OUT/'Table2_core_surface_water_results.csv',index=False)\nif terrain is not None: terrain.to_csv(OUT/'Supplementary_Table_S4_terrain_statistics.csv',index=False)\nif wc is not None:\n    c=wc[wc['zone'].eq('Cumulative temporary flood')].copy()\n    c['share_pct']=c['area_km2']/c['area_km2'].sum()*100\n    c.sort_values('area_km2',ascending=False).to_csv(OUT/'Supplementary_Table_S5_WorldCover.csv',index=False)\nif ghsl is not None:\n    ghsl.to_csv(OUT/'GHSL_built_exposure_summary.csv',index=False)\nif mndwi is not None: mndwi.to_csv(OUT/'Supplementary_Table_S6_MNDWI_sensitivity.csv',index=False)\nif monthly is not None: monthly.sort_values('month').to_csv(OUT/'CHIRPS_monthly_2023.csv',index=False)\n\npd.DataFrame({\n 'quantity':['Study area','Event temporary flood','Cumulative temporary flood','Recurrent temporary flood','Sporadic component','Surface-change zone'],\n 'expected_km2':[8938.927,5.868,7.386,4.487,2.899,2032.146]\n}).to_csv(OUT/'manuscript_expected_values.csv',index=False)\nprint('Final tables saved.')\n